**Imports**

In [ ]:
from google.colab import drive
import os
import pandas as pd

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import numpy as np

In [ ]:
from google.colab import userdata #to get the openAI API key
import json
import time
from tqdm import tqdm

In [ ]:
from openai import OpenAI

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
!pip install sacrebleu

  Using cached sacrebleu-2.5.1-py3-none-any.whl.metadata (51 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached colorama-0.4.6-py2.py3-none-any.whl.metadata (17 kB)
Using cached sacrebleu-2.5.1-py3-none-any.whl (104 kB)
Using cached colorama-0.4.6-py2.py3-none-any.whl (25 kB)
Using cached portalocker-3.2.0-py3-none-any.whl (22 kB)


In [ ]:
from sacrebleu.metrics import BLEU

In [ ]:
!pip install unbabel-comet==2.2.7

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.0/91.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.4/101.4 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 13.9 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not current

In [ ]:
from comet import download_model, load_from_checkpoint

In [ ]:
import time
import re

# **Dataset**

In [ ]:
df = pd.read_excel('/content/drive/MyDrive/Colab Notebooks/WikiEn-Ara.xlsx')
df.head()

,id,english_text,arabic_text,Field,Link
0,1,William Barley (1565–1614) was an English book...,ويليام بارلي (1565 – 1614). كان بائع كُتب وناش...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/William_Barley
1,2,"At this time, Cantillon became involved with B...",في هذا الوقت، أصبح كانتيلون مشاركاً مع التجاري...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Richard_Cantillon
2,3,"Harold Adams Innis FRSC (November 5, 1894 – No...",هارولد آدم اينيس (5 نوفمبر 1894- 8 نوفمبر 195...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Harold_Innis
3,4,"In computer science, binary search, also known...",في علم الحاسوب، خوارزمية البحث الثنائي ، المعر...,Computing,https://en.wikipedia.org/wiki/Binary_search_al...
4,5,The Manchester Mark 1 was one of the earliest ...,مانشتر مارك 1‏ كان واحدًا من أوائل حواسيب البر...,Computing,https://en.wikipedia.org/wiki/Manchester_Mark_1


In [ ]:
for text in df['english_text']:
  len += len(text.split())
print(len)

TypeError: 'int' object is not callable

In [ ]:
#drop the id column
df = df.set_index('id')
df.head()

,english_text,arabic_text,Field,Link
id,,,,
1,William Barley (1565–1614) was an English book...,ويليام بارلي (1565 – 1614). كان بائع كُتب وناش...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/William_Barley
2,"At this time, Cantillon became involved with B...",في هذا الوقت، أصبح كانتيلون مشاركاً مع التجاري...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Richard_Cantillon
3,"Harold Adams Innis FRSC (November 5, 1894 – No...",هارولد آدم اينيس (5 نوفمبر 1894- 8 نوفمبر 195...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Harold_Innis
4,"In computer science, binary search, also known...",في علم الحاسوب، خوارزمية البحث الثنائي ، المعر...,Computing,https://en.wikipedia.org/wiki/Binary_search_al...
5,The Manchester Mark 1 was one of the earliest ...,مانشتر مارك 1‏ كان واحدًا من أوائل حواسيب البر...,Computing,https://en.wikipedia.org/wiki/Manchester_Mark_1


In [ ]:
df = df.rename(columns={'arabic_text':'arabic_reference'})
df.head()

,english_text,arabic_reference,Field,Link
id,,,,
1,William Barley (1565–1614) was an English book...,ويليام بارلي (1565 – 1614). كان بائع كُتب وناش...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/William_Barley
2,"At this time, Cantillon became involved with B...",في هذا الوقت، أصبح كانتيلون مشاركاً مع التجاري...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Richard_Cantillon
3,"Harold Adams Innis FRSC (November 5, 1894 – No...",هارولد آدم اينيس (5 نوفمبر 1894- 8 نوفمبر 195...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Harold_Innis
4,"In computer science, binary search, also known...",في علم الحاسوب، خوارزمية البحث الثنائي ، المعر...,Computing,https://en.wikipedia.org/wiki/Binary_search_al...
5,The Manchester Mark 1 was one of the earliest ...,مانشتر مارك 1‏ كان واحدًا من أوائل حواسيب البر...,Computing,https://en.wikipedia.org/wiki/Manchester_Mark_1


**Information about the Dataset**

In [ ]:
len(df) #number of texts
unique_fields = df['Field'].unique()
print(unique_fields)
len(unique_fields)

['Business, economics, and finance biographies' 'Computing'
 'Culture and Society' 'Culture and society biographies ' 'Education '
 'Engineering and technology ' 'Health and medicine '
 'Heraldry, honors, and vexillology ' 'History ' 'Law' 'Law biographies '
 'Literature and theatre ' 'Autobiographies and memoirs '
 ' Magazines and comics ' 'Novels, including graphic novels '
 'Mathematics ' 'Biographies of mathematicians ' 'Media']


18

In [ ]:
field_counts = df['Field'].value_counts()
print(field_counts)
print(sum(field_counts.values))

Field
History                                         65
Culture and Society                             26
Novels, including graphic novels                19
Engineering and technology                      15
Culture and society biographies                 14
Autobiographies and memoirs                      8
Health and medicine                              6
Literature and theatre                           6
Education                                        5
Media                                            4
 Magazines and comics                            4
Computing                                        4
Law biographies                                  4
Business, economics, and finance biographies     3
Law                                              3
Heraldry, honors, and vexillology                2
Mathematics                                      1
Biographies of mathematicians                    1
Name: count, dtype: int64
190


Average number of words for English texts - Average number of words for Arabic texts - Standard deviation for both - Minimum and maximum lengths for both - Total word count for the entire corpus for both

In [ ]:
df['english_word_count'] = df['english_text'].str.split().str.len()
df['arabic_word_count'] = df['arabic_reference'].str.split().str.len()
df

,english_text,arabic_reference,Field,Link,english_word_count,arabic_word_count
id,,,,,,
1,William Barley (1565–1614) was an English book...,ويليام بارلي (1565 – 1614). كان بائع كُتب وناش...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/William_Barley,131,111
2,"At this time, Cantillon became involved with B...",في هذا الوقت، أصبح كانتيلون مشاركاً مع التجاري...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Richard_Cantillon,131,108
3,"Harold Adams Innis FRSC (November 5, 1894 – No...",هارولد آدم اينيس (5 نوفمبر 1894- 8 نوفمبر 195...,"Business, economics, and finance biographies",https://en.wikipedia.org/wiki/Harold_Innis,76,87
4,"In computer science, binary search, also known...",في علم الحاسوب، خوارزمية البحث الثنائي ، المعر...,Computing,https://en.wikipedia.org/wiki/Binary_search_al...,211,197
5,The Manchester Mark 1 was one of the earliest ...,مانشتر مارك 1‏ كان واحدًا من أوائل حواسيب البر...,Computing,https://en.wikipedia.org/wiki/Manchester_Mark_1,117,109
...,...,...,...,...,...,...
186,Georg Ferdinand Ludwig Philipp Cantor was a ma...,غيورغ فرديناند لودفيغ فيليب كانتور عاش ما بين ...,Biographies of mathematicians,https://en.wikipedia.org/wiki/Georg_Cantor,97,90
187,"""From the Doctor to My Son Thomas"" is a viral ...",من الطبيب إلى إبني توماس هو فيديو مسجل من المم...,Media,https://en.wikipedia.org/wiki/From_the_Doctor_...,86,77
188,Street newspapers (or street papers) are newsp...,جرائد الشارع (أو صحف الشارع) هي جرائد ومجلات ي...,Media,https://en.wikipedia.org/wiki/Street_newspaper,182,153


In [ ]:
statistics = {
    'English': {
        'total': df['english_word_count'].sum(),
        'min': df['english_word_count'].min(),
        'max': df['english_word_count'].max(),
        'mean': df['english_word_count'].mean(),
        'std': df['english_word_count'].std() },
    'Arabic': {
        'total': df['arabic_word_count'].sum(),
        'min': df['arabic_word_count'].min(),
        'max': df['arabic_word_count'].max(),
        'mean': df['arabic_word_count'].mean(),
        'std': df['arabic_word_count'].std()}}
statistics

{'English': {'total': 22126,
  'min': 30,
  'max': 259,
  'mean': 116.45263157894736,
  'std': 47.408339502811245},
 'Arabic': {'total': 19955,
  'min': 30,
  'max': 264,
  'mean': 105.02631578947368,
  'std': 45.018565742225164}}

In [ ]:
train_df, test_df= train_test_split(df, test_size=0.2, random_state=42)
print(train_df.shape[0])
print(test_df.shape[0])

152
38


# **Systems Under Evaluation**

# **Large Language Model (GPT 5.0 mini)**

In [ ]:
openAI_Key = userdata.get('NMT_GPT5mini')

In [ ]:
client = OpenAI(api_key=openAI_Key)

In [ ]:
system_prompt = """
You are an expert translator specializing in English to Modern Standard Arabic translations. Your goal is to produce high-quality, accurate translations that are culturally appropriate, stylistically natural, and aligned with professional standards as per ISO 17100.
Purpose: Produce translations of encyclopedic Wikipedia excerpts for evaluation in a machine translation research.
Target Audience: Arabic-speaking general readers.
Style: Encyclopedic.
Language Pair: English to Modern Standard Arabic.
Domain Context: The text is from a Wikipedia article in the field of {field}.
Here are five high-quality example translations to guide your output:
Example 1 (Field: History):
English: The Palace of Versailles is a royal château in Versailles, Yvelines, in the Île-de-France region of France. When the château was built, Versailles was a country village; today, however, it is a suburb of Paris, some 20 kilometres southwest of the French capital. The court of Versailles was the centre of political power in France from 1682, when Louis XIV moved from Paris, until the royal family was forced to return to the capital in October 1789 after the beginning of the French Revolution. Versailles is therefore famous not only as a building, but as well as a symbol of the system of absolute monarchy of the Ancien Régime.
Arabic: قصر فرساي هو قلعة ملكية تقع في فرساي في منطقة إيل دو فرانس الفرنسية. عندما بُنيت القلعة، كانت فرساي مجرد قرية؛ ولكن اليوم تُعتبر فرساي من ضواحي باريس وتبعد حوالي عشرين كيلومترًا عن العاصمة الفرنسية. كان بلاط فرساي مركزًا للسلطة السياسية في فرنسا من عام 1682، عندما قرر لويس الرابع عشر الانتقال من باريس، حتى أجبِرت العائلة الملكية على العودة إلى العاصمة في أكتوبر 1789 بعد بداية الثورة الفرنسية. لا تُعد فرساي مجرد مبنى شهير بل رمزًا لنظام الملكية المطلقة للنظام القديم
Example 2 (Field: Law):
English: The Coinage Act of 1965, Pub. L. 89–81, 79 Stat. 254, enacted July 23, 1965, eliminated silver from the circulating United States dime (ten-cent piece) and quarter dollar coins. It also reduced the silver content of the half dollar from 90 percent to 40 percent; silver in the half dollar was subsequently eliminated by a 1970 law.
Arabic: قانون العملة لعام 1965، 254، صدر في 23 يوليو/تموز 1965، وألغى الفضة من العُملات المعدنية فئة العشرة سنتات و الربع دولار المتداولة في الولايات المتحدة. كما أدى ذلك إلى خفض محتوى الفضة في نصف الدولار من 90 في المائة إلى 40 في المائة؛ وألغى بعد ذلك الفضة في نصف الدولار بموجب قانون صدر عام 1970
Example 3 (Field: Magazines and comics):
English: Alice in the Country of Hearts is a Japanese female-oriented visual novel developed by Quin Rose. The game is a re-imagining of Lewis Carroll's classic 1865 novel Alice's Adventures in Wonderland. There are multiple sequel games, as well as multiple manga series, licensed in North America originally by Tokyopop and later by Yen Press and Seven Seas Entertainment. An original video animation adaptation was announced for release in November 2008, but was later delayed. Instead, an anime film adaptation produced by Asahi Production was released in Japanese theaters in July 2011.
Arabic: أليس في بلاد القلوب هي رواية مرئية يابانية مغامرة رومانسية موجهة للإناث، طورت من قِبل كوين روز. اللعبة هي إعادة تخيل لرواية لويس كارول الكلاسيكية: مغامرات أليس في بلاد العجائب. كانت هناك عدة تتمات للعبة، فضلا عن العديد من سلاسل مانغا. فيديو رسوم متحركة أصلي أعلن عن موعد إصداره في نوفمبر 2008، ولكن تم تأجيل الإطلاق في وقت لاحق. فيلم أنمي من إنتاج اساهي برودكشن صدر في المسارح اليابانية في يوليو 2011
Example 4 (Field: Novels)
English: A debut novel is the first novel a novelist publishes. Debut novels are often the author's first opportunity to make an impact on the publishing industry, and thus the success or failure of a debut novel can affect the ability of the author to publish in the future. Sometimes new novelists will self-publish their debut novels, because publishing houses will not risk the capital needed to market books by an unknown author to the public.
Arabic: الرواية الأولى هي أول عمل روائي منشور للكاتب، وغالبًا ما تُشكّل الفرصة الأولى له لإثبات حضوره في عالم النشر. يمكن لنجاح أو فشل هذه الرواية أن يؤثر بشكل مباشر على قدرة الكاتب على الاستمرار في النشر مستقبلاً. في كثير من الحالات، يلجأ الروائيون الجدد إلى نشر رواياتهم الأولى بأنفسهم، نظرًا لتردد دور النشر في المخاطرة بالاستثمار في أعمال لكتاب غير معروفي
Example 5 (Field: Education)
English: The Education Program for Gifted Youth (EPGY) at Stanford University was a loose collection of gifted education programs formerly located within Stanford Pre-Collegiate Studies program.[1] EPGY included distance and residential summer courses for students of all ages. Many of the courses were distance learning, meaning that courses were taught remotely via the Internet, rather than in the traditional classroom setting. Courses targeted students from elementary school up to advanced college graduate. Subjects offered included: Mathematics, English, Humanities, Physics, and Computer Science. Stanford Pre-Collegiate Studies is similar to the Center for Talented Youth at the Johns Hopkins University in terms of certain objectives.
Arabic: برنامج التعليم للشباب الموهوبين (EPGY) في جامعة ستانفورد، هو مجموعة فضفاضة من برامج التعليم للموهوبين التي كانت موجودة سابقا في دراسات ستانفورد قبل الجامعية في جامعة ستانفورد. وشملت برامج إبغي دورات المسافة والصيف السكنية للطلاب من جميع الأعمار. وكان العديد من الدورات التعلم عن بعد، وهذا يعني أن الدورات تدرس عن بعد عن طريق الإنترنت، وليس في إعدادالفصول الدراسية التقليدية. دورات استهدفت الطلاب من المدرسة الابتدائية حتى خريج الكلية المتقدمة. الموضوعات المقدمة شملت: الرياضيات، واللغة الإنجليزية، والعلوم الإنسانية، والفيزياء، وعلوم الكمبيوتر. تشبه دراسات ستانفورد ما قبل الجامعية مركز الشباب الموهوبين في جامعة جونز هوبكنز من حيث الأهداف المعينة
Output only the Arabic translation without any additional explanations or text.
"""

In [ ]:
def gpt_translate(text_english, text_field):
    gpt_prompt = system_prompt.format(field=text_field)
    try:
        response = client.chat.completions.create(
            model="gpt-5-mini",
            messages=[
                {"role": "system", "content": gpt_prompt},
                {"role": "user", "content": text_english}
            ],
            max_completion_tokens=4096,  # Changed from max_tokens
            n=1,
            stop=None,
        )
        arabic_translation = response.choices[0].message.content.strip()
        return arabic_translation
    except Exception as e:
        print("Error during translation:", str(e))
        return None

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)  # Stratify by Field if possible

In [ ]:
for split in [train_df, test_df]:split['gpt_translation'] = np.nan

In [ ]:
'''for index, row in tqdm(train_df.iterrows(), total = len(train_df), desc="Translating train_df"):
  translation = gpt_translate(row['english_text'], row['Field'])
  train_df.at[index, 'gpt_translation'] = translation
  if translation is None:
        print(f"Failed translation for ID {index}")'''

'for index, row in tqdm(train_df.iterrows(), total = len(train_df), desc="Translating train_df"):\n  translation = gpt_translate(row[\'english_text\'], row[\'Field\'])\n  train_df.at[index, \'gpt_translation\'] = translation\n  if translation is None:\n        print(f"Failed translation for ID {index}")'

In [ ]:
'''output_path = '/content/drive/MyDrive/Colab Notebooks/train_df_translated.xlsx'
train_df.to_csv(output_path, index=True)'''

"output_path = '/content/drive/MyDrive/Colab Notebooks/train_df_translated.xlsx'\ntrain_df.to_csv(output_path, index=True)"

In [ ]:
train_df_translated = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/train_df_translated (1).xlsx")
train_df_translated

,id,english_text,arabic_reference,Field,Link,english_word_count,arabic_word_count,gpt_translation
0,52,Some Thoughts Concerning Education is a 1693 t...,بعض الأفكار عن التعليم أطروحة كتبها الفيلسوف ...,Education,https://en.wikipedia.org/wiki/Some_Thoughts_Co...,181,166,«بعض الأفكار حول التربية» رسالة صدرت عام 1693 ...
1,36,Emily Wilding Davison (11 October 1872 – 8 Ju...,إميلي وايلدينغ دافيسون (11 أكتوبر 1872 - 8 ي...,Culture and society biographies,https://en.wikipedia.org/wiki/Emily_Davison,219,189,إميلي وايلدينغ دافيسون (11 October 1872 – 8 Ju...
2,119,"The Jarrow March of 5–31 October 1936, also kn...",مسيرة جارو امتدت من 5 حتى 31 أكتوبر عام 1936، ...,History,https://en.wikipedia.org/wiki/Jarrow_March,102,103,مسيرة جارو في الفترة من 5 إلى 31 أكتوبر 1936، ...
3,61,Project Rover was a United States project to d...,كان «مشروع روفر» مشروعًا تابعًا للولايات المتح...,Engineering and technology,https://en.wikipedia.org/wiki/Project_Rover,126,125,كان مشروع روفر مشروعًا أمريكيًا لتطوير صاروخ ن...
4,162,The Analytical Review was an English periodica...,المراجعة التحليلية هي مجلة دورية بريطانية، أسس...,Magazines and comics,https://en.wikipedia.org/wiki/Analytical_Review,59,39,كانت «المراجعة التحليلية» دورية إنجليزية نُشرت...
...,...,...,...,...,...,...,...,...
147,107,The Gurian Republic was an insurgent community...,كانت الجمهورية الغوريانية مجتمعًا متمردًا وُجد...,History,https://en.wikipedia.org/wiki/Gurian_Republic,99,82,كانت جمهورية غوريا كيانًا متمردًا قائمًا بين ع...
148,15,The ghost appeared to claim that Fanny had bee...,ظهر ذلك الشبح ليزعم بأن فاني قد تم تسميمها بما...,Culture and Society,NaN,120,132,ادعى الشبح أن فاني قد سُمِّمت بالزرنيخ، وأُثير...
149,93,"The Birmingham campaign, also known as the Bir...",حملة برمنجهام هي حركة نظمها مؤتمر القيادة المس...,History,https://en.wikipedia.org/wiki/Birmingham_campaign,83,64,حملة برمنغهام، المعروفة أيضًا بحركة برمنغهام أ...
150,180,The Penelopiad is a novella by Canadian author...,بينلوبياد هي رواية للكاتبة مارغريت أتوود تم نش...,"Novels, including graphic novels",https://en.wikipedia.org/wiki/The_Penelopiad,112,93,رواية قصيرة بعنوان «بينيلوبياد» للكاتبة الكندي...


In [ ]:
#we assess the quality of the train test before proceeding with the test set translation and human annotation
subset_df = train_df_translated[:38] #25% of the dataset
subset_df

,id,english_text,arabic_reference,Field,Link,english_word_count,arabic_word_count,gpt_translation
0,52,Some Thoughts Concerning Education is a 1693 t...,بعض الأفكار عن التعليم أطروحة كتبها الفيلسوف ...,Education,https://en.wikipedia.org/wiki/Some_Thoughts_Co...,181,166,«بعض الأفكار حول التربية» رسالة صدرت عام 1693 ...
1,36,Emily Wilding Davison (11 October 1872 – 8 Ju...,إميلي وايلدينغ دافيسون (11 أكتوبر 1872 - 8 ي...,Culture and society biographies,https://en.wikipedia.org/wiki/Emily_Davison,219,189,إميلي وايلدينغ دافيسون (11 October 1872 – 8 Ju...
2,119,"The Jarrow March of 5–31 October 1936, also kn...",مسيرة جارو امتدت من 5 حتى 31 أكتوبر عام 1936، ...,History,https://en.wikipedia.org/wiki/Jarrow_March,102,103,مسيرة جارو في الفترة من 5 إلى 31 أكتوبر 1936، ...
3,61,Project Rover was a United States project to d...,كان «مشروع روفر» مشروعًا تابعًا للولايات المتح...,Engineering and technology,https://en.wikipedia.org/wiki/Project_Rover,126,125,كان مشروع روفر مشروعًا أمريكيًا لتطوير صاروخ ن...
4,162,The Analytical Review was an English periodica...,المراجعة التحليلية هي مجلة دورية بريطانية، أسس...,Magazines and comics,https://en.wikipedia.org/wiki/Analytical_Review,59,39,كانت «المراجعة التحليلية» دورية إنجليزية نُشرت...
5,77,The 1880 United States presidential election w...,كانت الانتخابات الرئاسية في الولايات المتحدة ع...,History,https://en.wikipedia.org/wiki/1880_United_Stat...,46,71,كانت انتخابات الرئاسة الأمريكية لعام 1880 هي ا...
6,70,Icos Corporation (trademark ICOS) was an Ameri...,شركة إيكوس كانت سابقاً شركة أمريكية للتكنولوجي...,Health and medicine,https://en.wikipedia.org/wiki/Icos,101,83,شركة إيكوس (العلامة التجارية ICOS) كانت شركة أ...
7,30,The Whitechapel murders were committed in or n...,جرائم قتل وايت تشابل هي مجموعة من جرائم القتل ...,Culture and Society,https://en.wikipedia.org/wiki/Whitechapel_murders,190,182,وقعت جرائم قتل وايت تشابل في أو بالقرب من حي و...
8,129,Operation Grapple was a set of four series of ...,كانت عملية المرساة اسمًا لأربعة سلاسل اختبار أ...,History,https://en.wikipedia.org/wiki/Operation_Grapple,93,74,كانت عملية غراپل سلسلة مكوَّنة من أربع دفعات م...
9,153,Pilgrim at Tinker Creek is a 1974 nonfiction n...,الحج في خليج (نهر) تنكر هو كتاب سرد عام 1974 ق...,Literature and theatre,https://en.wikipedia.org/wiki/Pilgrim_at_Tinke...,86,80,الحاج في تينكر كريك هو كتاب سردي غير روائي صدر...


In [ ]:
translations = subset_df['gpt_translation'].tolist()
references = subset_df['arabic_reference'].tolist()
references


[' بعض الأفكار عن التعليم أطروحة كتبها الفيلسوف الإنجليزي جون لوك عام 1693 يناقش فيها «تعليم الفئة النبيلة». وقد ظلت هذه الأطروحة أهم الأعمال الفلسفية عن التعليم في إنجلترا لأكثر من قرن، وتُرجمت خلال القرن الثامن عشر إلى جميع اللغات الأوروبية المكتوبة الرئيسية تقريبًا، كما اعترف بتأثيرها البالغ الكتاب الأوروبيون الذين تناولوا موضوع التعليم بعد لوك، بمن فيهم جان جاك روسو. \nوفي مقالته التي تسمى «مقال عن الفهم الإنساني» الذي كُتب في عام 1690، ابتكر لوك نظرية عقلية جديدة، تؤمن النظرية بأن العقل عند مولد الإنسان وقبل تعلمه مثل الصفحة البيضاء، فهو لا يولد بأي أفكار فطرية أو غريزية. وفي «بعض الأفكار عن التعليم» يوضح لوك ثلاث طرق لتعليم هذا العقل وهم: تنمية جسد صحي والمحافظة عليه، تربية شخصية تتميز بالأخلاق الحميدة، واختيار مناهج دراسي مناسبة. \nفي البداية، كتب لوك هذا المقال على هيئة رسائل يبعثها لصديق له من الطبقة الأرسطوقراطية، ولكن نصائحه حازت على الرضى والإعجاب بسبب قيمه التعليمية التي سمحت للسيدات والطبقات الفقيرة بالارتفاع ليتساوو بالارسطوقراطيين الذي، في الأصل، كتب لوك هذا الكتاب من ا

In [ ]:
# Compute overall BLEU
bleu = BLEU()
bleu_score = bleu.corpus_score(translations, [references]).score
print(f"Overall BLEU score for subset: {bleu_score:.2f}")

Overall BLEU score for subset: 18.84


In [ ]:
data = [
    {"src": row['english_text'], "mt": row['gpt_translation'], "ref": row['arabic_reference']}
    for _, row in subset_df.iterrows()
]

In [ ]:
model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [ ]:
comet_scores = model.predict(data, batch_size=8, gpus=0)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
Predicting DataLoader 0: 100%|██████████| 5/5 [04:54<00:00, 58.89s/it]


In [ ]:
print(f"Overall COMET score for subset: {comet_scores.system_score:.4f}")

Overall COMET score for subset: 0.8579


In [ ]:
#save subset_df as excel sheet for annotators scoring
subset_df.to_excel('/content/drive/MyDrive/Colab Notebooks/subset_df.xlsx', index=False)

In [ ]:
test_df

,english_text,arabic_reference,Field,Link,english_word_count,arabic_word_count,gpt_translation
id,,,,,,,
176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels",https://en.wikipedia.org/wiki/The_Historian,81,79,NaN
181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels",https://en.wikipedia.org/wiki/The_Phantom_Toll...,107,95,NaN
112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History,https://en.wikipedia.org/wiki/History_of_Solid...,70,65,NaN
66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology,https://en.wikipedia.org/wiki/Sholes_and_Glidd...,115,90,NaN
102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History,https://en.wikipedia.org/wiki/Ehime_Maru_and_U...,99,82,NaN
16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society,https://en.wikipedia.org/wiki/Cottingley_Fairies,196,169,NaN
10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society,https://en.wikipedia.org/wiki/Burke_and_Hare_m...,236,264,NaN
17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society,https://en.wikipedia.org/wiki/Death_of_Mark_Sa...,180,182,NaN
142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law,https://en.wikipedia.org/wiki/Coinage_Act_of_1...,100,82,NaN


In [ ]:
#train the whole test set
for index, row in tqdm(test_df.iterrows(), total = len(test_df), desc="Translating test set"):
  translation = gpt_translate(row['english_text'], row['Field'])
  test_df.at[index, 'gpt_translation'] = translation
  if translation is None:
        print(f"Failed translation for ID {index}")

Translating test set:   0%|          | 0/38 [00:00<?, ?it/s]/tmp/ipython-input-767924103.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'المؤرخ هي الرواية الأولى للكاتبة الأمريكية إليزابيث كوستوفا، ونُشرت عام 2005. تدمج حبكتها بين التاريخ والفولكلور المتعلق بفلاد المخوزق ونظيره الخيالي الكونت دراكولا. كان والد كوستوفا يروي لها قصصًا عن دراكولا عندما كانت طفلة، ولاحقًا ألهمتها تلك التجربة لتحويلها إلى رواية. عملت على الكتاب لمدة عشر سنوات ثم باعتها خلال بضعة أشهر لدار ليتل براون آند كومباني، التي اشترت حقوقها بمبلغ قدره مليونا دولار أمريكي.' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  test_df.at[index, 'gpt_translation'] = translation
Translating test set: 100%|██████████| 38/38 [14:44<00:00, 23.27s/it]


In [ ]:
output_path = '/content/drive/MyDrive/Colab Notebooks/test_df_translated(1).xlsx'
test_df.to_excel(output_path, index=True)

In [ ]:
test_df_translated = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/test_df_translated(1).xlsx")
test_df_translated

,id,english_text,arabic_reference,Field,Link,english_word_count,arabic_word_count,gpt_translation
0,176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels",https://en.wikipedia.org/wiki/The_Historian,81,79,NaN
1,181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels",https://en.wikipedia.org/wiki/The_Phantom_Toll...,107,95,NaN
2,112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History,https://en.wikipedia.org/wiki/History_of_Solid...,70,65,NaN
3,66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology,https://en.wikipedia.org/wiki/Sholes_and_Glidd...,115,90,NaN
4,102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History,https://en.wikipedia.org/wiki/Ehime_Maru_and_U...,99,82,NaN
5,16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society,https://en.wikipedia.org/wiki/Cottingley_Fairies,196,169,NaN
6,10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society,https://en.wikipedia.org/wiki/Burke_and_Hare_m...,236,264,NaN
7,17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society,https://en.wikipedia.org/wiki/Death_of_Mark_Sa...,180,182,NaN
8,142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law,https://en.wikipedia.org/wiki/Coinage_Act_of_1...,100,82,NaN
9,125,"The murder of Julia Martha Thomas, dubbed the ...",إن مقتل جوليا مارثا توماس ، يطلق عليه اسم لغز ...,History,https://en.wikipedia.org/wiki/Murder_of_Julia_...,142,138,NaN


In [ ]:
translations_test = test_df_translated['gpt_translation'].tolist()
references_test = test_df_translated['arabic_reference'].tolist()

In [ ]:
# Compute overall BLEU
bleu = BLEU()
# Filter out non-string translations
translations_test_filtered = [t for t in translations_test if isinstance(t, str)]
references_test_filtered = [references_test[i] for i, t in enumerate(translations_test) if isinstance(t, str)]
bleu_score = bleu.corpus_score(translations_test_filtered, [references_test_filtered]).score
print(f"Overall BLEU score for subset: {bleu_score:.2f}")

Overall BLEU score for subset: 17.73


In [ ]:
data_test = [
    {"src": row['english_text'], "mt": row['gpt_translation'], "ref": row['arabic_reference']}
    for _, row in test_df_translated.iterrows()
]

In [ ]:
comet_scores_test = model.predict(data_test, batch_size=8, gpus=0)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
Predicting DataLoader 0: 100%|██████████| 5/5 [05:11<00:00, 62.27s/it]


In [ ]:
print(f"Overall COMET score for test set: {comet_scores_test.system_score:.4f}")

Overall COMET score for test set: 0.8649


# **AI Agent**

In [ ]:
openAI_Key = userdata.get('NMT_GPT5mini')

In [ ]:
client = OpenAI(api_key=openAI_Key)

we drop the arabic reference from the test set to keep the agent self contained

In [ ]:
test_df_agent = test_df.drop(columns=['Link','english_word_count','arabic_word_count'])
test_df_agent

,english_text,arabic_reference,Field
id,,,
176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels"
181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels"
112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History
66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology
102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History
16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society
10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society
17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society
142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law


In [ ]:
path = '/content/drive/MyDrive/Colab Notebooks/test_df_agent.xlsx'
test_df_agent.to_excel(path, index=True)

In [ ]:
test_df_agent = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/test_df_agent.xlsx")
test_df_agent

,id,english_text,arabic_reference,Field
0,176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels"
1,181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels"
2,112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History
3,66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology
4,102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History
5,16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society
6,10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society
7,17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society
8,142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law
9,125,"The murder of Julia Martha Thomas, dubbed the ...",إن مقتل جوليا مارثا توماس ، يطلق عليه اسم لغز ...,History


In [ ]:
with open('/content/drive/MyDrive/Colab Notebooks/Translator_Prompt.txt') as file:
  translator_prompt = file.read()
translator_prompt

"You are an expert translator specializing in English to Modern Standard Arabic translations. Your goal is to produce high-quality, accurate translations that are culturally appropriate, stylistically natural, and aligned with professional standards as per ISO 17100.\nPurpose: Produce translations of encyclopedic Wikipedia excerpts for evaluation in a machine translation research.\nTarget Audience: Arabic-speaking general readers.\nStyle: Encyclopedic.\nLanguage Pair: English to Modern Standard Arabic.\nDomain Context: The text is from a Wikipedia article in the field of {field}.\nHere are five high-quality example translations to guide your output:\nExample 1 (Field: History):\nEnglish: The Palace of Versailles is a royal château in Versailles, Yvelines, in the Île-de-France region of France. When the château was built, Versailles was a country village; today, however, it is a suburb of Paris, some 20 kilometres southwest of the French capital. The court of Versailles was the centre o

In [ ]:
with open('/content/drive/MyDrive/Colab Notebooks/Evaluator_Prompt.txt') as file:
  evaluator_prompt = file.read()
evaluator_prompt

'You are an expert translator specializing in English to Modern Standard Arabic translations. Your task is to evaluate LLM-generated translations of encyclopedic Wikipedia excerpts for evaluation in a machine translation research. You must assess these translations on:\n1. Adequacy: How well does the translation convey the meaning of the English source text? (5=perfect, 1=unacceptable)\n2. Fluency: How natural and grammatically correct is the Arabic translation for a native Arabic speaker? (5=perfect, 1=unacceptable)\n3. Style: How well does the translation maintain the encyclopedic/Wikipedia style and tone of the source? (5=perfect, 1=unacceptable)\nProvide:\nA score from 1 to 5 for each metric.\n2-3 specific improvements (e.g., "Replace term X with Y for accuracy").\nInput: English Text: {english_text} Translation: {translation} Field: {field}\nOutput format: \nAdequacy: [score]/5 \nFluency: [score]/5 \nStyle: [score]/5 \nImprovements:\n[Improvement 1]\n[Improvement 2, if applicable]

In [ ]:
with open('/content/drive/MyDrive/Colab Notebooks/Refiner_Prompt.txt') as file:
  refiner_prompt = file.read()
refiner_prompt

'You are an expert translator specializing in English to Modern Standard Arabic translations. Your task is to refine the provided Arabic translation based on the improvements below, ensuring high adequacy, fluency, and encyclopedic style suitable for Arabic Wikipedia. Focus only on the suggested improvements without additional text.\nInput: Original Translation: {translation} Critique: {critique} Field: {field}\nOutput only the refined Arabic translation.'

In [ ]:
# Translate
def translate(text_id, text_english, text_field):
    translator_prompt_with_field = translator_prompt.format(field=text_field)
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": translator_prompt_with_field},
                {"role": "user", "content": text_english}
            ],
            temperature=0.3,
            max_tokens=1000
        )
        arabic_translation = response.choices[0].message.content.strip()
        if not arabic_translation:
            print(f"Empty translation for text ID: {text_id}")
            return None
        return arabic_translation
    except Exception as e:
        print(f"Translation error for text ID: {text_id}: {e}")
        return None

In [ ]:
# Evaluate
def evaluate(text_id, text_english, text_field, arabic_translation):
    if arabic_translation is None:
        return None, {'Adequacy': None, 'Fluency': None, 'Style': None}
    scores = {'Adequacy': None, 'Fluency': None, 'Style': None}
    prompt = evaluator_prompt.format(english_text=text_english, translation=arabic_translation, field=text_field)
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": "Evaluate the translation."}
            ],
            temperature=0.3,
            max_tokens=500
        )
        eval_text = response.choices[0].message.content.strip()

        # Parse evaluation output
        score_pattern = r"(Adequacy|Fluency|Style): (\d)/5"
        improvement_pattern = r"Improvements:\n([\s\S]*)"
        scores_match = re.findall(score_pattern, eval_text)
        improvements_match = re.search(improvement_pattern, eval_text)

        for metric, score in scores_match:
            scores[metric] = int(score)
        improvements = improvements_match.group(1).strip().split('\n') if improvements_match else []
        return eval_text, scores
    except Exception as e:
        print(f"Evaluation error for text ID: {text_id}: {e}")
        return None, scores

In [ ]:
# Refine
def refine(text_id, arabic_translation, eval_text, text_field):
    if arabic_translation is None or eval_text is None:
        return arabic_translation
    prompt = refiner_prompt.format(translation=arabic_translation, critique=eval_text, field=text_field)
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": "Refine the translation."}
            ],
            temperature=0.3,
            max_tokens=1000
        )
        refined_translation = response.choices[0].message.content.strip()
        if not refined_translation:
            print(f"Empty refinement for text ID: {text_id}")
            return arabic_translation
        return refined_translation
    except Exception as e:
        print(f"Refinement error for text ID: {text_id}: {e}")
        return arabic_translation

In [ ]:
# Translate -> Evaluate (with scores) -> Refine translation if needed
def translate_evaluate_assess(text_id, text_english, text_field):
    arabic_translation = translate(text_id, text_english, text_field)
    if arabic_translation is None:
        return None, {'Adequacy': None, 'Fluency': None, 'Style': None}
    eval_text, scores = evaluate(text_id, text_english, text_field, arabic_translation)
    refined_translation = refine(text_id, arabic_translation, eval_text, text_field)
    return refined_translation, scores #scores from the evaluation step

In [ ]:
test_df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/test_df_agent.xlsx")
test_df['agent_translation'] = None
test_df['adequacy_score'] = None
test_df['fluency_score'] = None
test_df['style_score'] = None

In [ ]:
test_df

,id,english_text,arabic_reference,Field,agent_translation,adequacy_score,fluency_score,style_score
0,176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels",None,None,None,None
1,181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels",None,None,None,None
2,112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History,None,None,None,None
3,66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology,None,None,None,None
4,102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History,None,None,None,None
5,16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society,None,None,None,None
6,10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society,None,None,None,None
7,17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society,None,None,None,None
8,142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law,None,None,None,None
9,125,"The murder of Julia Martha Thomas, dubbed the ...",إن مقتل جوليا مارثا توماس ، يطلق عليه اسم لغز ...,History,None,None,None,None


In [ ]:
for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Processing test set"):
    refined_translation, scores = translate_evaluate_assess(idx, row['english_text'], row['Field'])
    test_df.at[idx, 'agent_translation'] = refined_translation
    if scores:
        test_df.at[idx, 'adequacy_score'] = scores['Adequacy']
        test_df.at[idx, 'fluency_score'] = scores['Fluency']
        test_df.at[idx, 'style_score'] = scores['Style']
    if refined_translation is None:
        print(f"Failed translation for ID: {idx}")
    time.sleep(1)

Processing test set: 100%|██████████| 38/38 [06:34<00:00, 10.38s/it]


In [ ]:
output_path = '/content/drive/MyDrive/Colab Notebooks/test_df_agent_translated.csv'
test_df.to_csv(output_path, index=True)

In [ ]:
test_df_translated = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/test_df_agent_translated.csv')
test_df_translated

,Unnamed: 0,id,english_text,arabic_reference,Field,agent_translation,adequacy_score,fluency_score,style_score
0,0,176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels",المؤرخ هي الرواية الأولى للكاتبة الأمريكية إلي...,5,5,5
1,1,181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels",بوابة الأشباح هي رواية مغامرات خيالية للأطفال ...,4,4,4
2,2,112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History,"تأسست ""التضامن""، وهي نقابة عمالية غير حكومية ب...",5,5,5
3,3,66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology,كانت آلة الكتابة من نوع شولز وغليدن (المعروفة ...,5,5,5
4,4,102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History,في 9 فبراير 2001، وعلى بعد حوالي تسعة أميال بح...,5,5,5
5,5,16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society,تظهر جنيات كوتينغلي في سلسلة من خمس صور فوتوغر...,4,4,4
6,6,10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society,كانت جرائم بيرك وهير سلسلة من ستة عشر جريمة قت...,5,5,5
7,7,17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society,مارك سوندرز كان محاميًا بريطانيًا تم إطلاق الن...,5,5,5
8,8,142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law,قانون العملة لعام 1873 أو قانون سك العملات لعا...,5,5,5
9,9,125,"The murder of Julia Martha Thomas, dubbed the ...",إن مقتل جوليا مارثا توماس ، يطلق عليه اسم لغز ...,History,كانت جريمة قتل جوليا مارثا توماس، التي أطلق عل...,5,5,5


In [ ]:
test_df_translated

,Unnamed: 0,id,english_text,arabic_reference,Field,agent_translation,adequacy_score,fluency_score,style_score
0,0,176,The Historian is the 2005 debut novel of Ameri...,هي الرواية الأولى للكاتبة الأمريكية إليزابيث ك...,"Novels, including graphic novels",المؤرخ هي الرواية الأولى للكاتبة الأمريكية إلي...,5,5,5
1,1,181,The Phantom Tollbooth is a children's fantasy ...,المحصل فانتوم هي رواية مغامرات خيالية للأطفال ...,"Novels, including graphic novels",بوابة الأشباح هي رواية مغامرات خيالية للأطفال ...,4,4,4
2,2,112,"Solidarity, a Polish non-governmental trade un...",حركة تضامن هي نقابة عمالية غير حكومية أسسها لي...,History,"تأسست ""التضامن""، وهي نقابة عمالية غير حكومية ب...",5,5,5
3,3,66,The Sholes and Glidden typewriter (also known ...,آلة شولز وغلايدن الكاتبة هي أول الآلات الكاتبة...,Engineering and technology,كانت آلة الكتابة من نوع شولز وغليدن (المعروفة ...,5,5,5
4,4,102,"On 9 February 2001, about nine nautical miles ...",في 9 فبراير 2001 اصطدمت الغواصة يو إس إس غرينف...,History,في 9 فبراير 2001، وعلى بعد حوالي تسعة أميال بح...,5,5,5
5,5,16,The Cottingley Fairies appear in a series of f...,تظهر جنيات كوتينجلي في سلسلة من خمس صور التقط...,Culture and Society,تظهر جنيات كوتينغلي في سلسلة من خمس صور فوتوغر...,4,4,4
6,6,10,The Burke and Hare murders were a series of si...,ارتكب بورك وهير 16 جريمة قتل موزعة على فترات ا...,Culture and Society,كانت جرائم بيرك وهير سلسلة من ستة عشر جريمة قت...,5,5,5
7,7,17,Mark Saunders was a British barrister who was ...,كان مارك سوندرز محاميًا بريطانيًا حين قُتل برص...,Culture and Society,مارك سوندرز كان محاميًا بريطانيًا تم إطلاق الن...,5,5,5
8,8,142,The Coinage Act of 1873 or Mint Act of 1873 wa...,كان قانون سك العملة لعام 1873 أو قانون العملات...,Law,قانون العملة لعام 1873 أو قانون سك العملات لعا...,5,5,5
9,9,125,"The murder of Julia Martha Thomas, dubbed the ...",إن مقتل جوليا مارثا توماس ، يطلق عليه اسم لغز ...,History,كانت جريمة قتل جوليا مارثا توماس، التي أطلق عل...,5,5,5


Firs, let us compute the overall adequacy, fluency, and style scores of the test data and compare them to human annotated scores

In [ ]:
average_adequacy = test_df_translated['adequacy_score'].mean()
average_fluency = test_df_translated['fluency_score'].mean()
average_style = test_df_translated['style_score'].mean()
print(f"Average adequacy score: {average_adequacy}")
print(f"Average fluency score: {average_fluency}")
print(f"Average style score: {average_style}")


Average adequacy score: 4.815789473684211
Average fluency score: 4.815789473684211
Average style score: 4.815789473684211


In [ ]:
translations = test_df_translated['agent_translation'].tolist()
references = test_df_translated['arabic_reference'].tolist()
references

['هي الرواية الأولى للكاتبة الأمريكية إليزابيث كوستوفا، نشرت في عام 2005. المؤرخ تمزج الرواية بين التاريخ الحقيقي وبين الفولكلور الشعبي عن فلاد شيبي، وهو المعروف في الحكايات الخيالية باسم الكونت دراكولا. استلهمت كوستوفا هذه الرواية من القصص التي كان أبوها يرويها لها في طفولتها عن دراكولا، وحين كبرت قررت أن تكتب رواية عن تلك القصص. وعملت على تأليف الكتاب لمدة عشر سنوات، ثم باعته بعد بضعة أشهر لدار النشر الأمريكية «ليتل وبراون وشركاه»، في مقابل 2 مليون دولار أمريكي ',
 'المحصل فانتوم هي رواية مغامرات خيالية للأطفال كتبها نورتون جاستر ، مع رسوم توضيحية لجولز فايفر ، نُشرت لأول مرة في عام 1961. تتبع القصة صبيًا صغيرًا يشعر بالملل يُدعى ميلو يتلقى بشكل غير متوقع كشكًا سحريًا للجباية ينقله إلى مملكة الحكمة التي كانت مزدهرة، ولكنها الآن مضطربة. جنبا إلى جنب مع كلب اسمه توك ومعهم الهراء، يذهب ميلو في مهمة بحثية عن القلعة في الهواء بحثًا عن أميرات المملكة المنفيتين، وهما قافية و منطق. بينما يتعلم ميلو دروسًا قيمة، يجد حب التعلم في قصة مليئة بالتورية والتلاعب بالألفاظ، مثل استكشاف المعاني الحرفي

In [ ]:
bleu = BLEU()
bleu_score = bleu.corpus_score(translations, [references]).score
print(f"Overall BLEU score for subset: {bleu_score:.2f}")

Overall BLEU score for subset: 22.83


In [ ]:
data = [
    {"src": row['english_text'], "mt": row['agent_translation'], "ref": row['arabic_reference']}
    for _, row in test_df_translated.iterrows()]

In [ ]:
model_path = download_model("Unbabel/wmt22-comet-da")
model = load_from_checkpoint(model_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [ ]:
comet_scores = model.predict(data, batch_size=8, gpus=0)

INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
Predicting DataLoader 0: 100%|██████████| 5/5 [05:59<00:00, 71.82s/it]


In [ ]:
print(f"Overall COMET score for subset: {comet_scores.system_score:.4f}")

Overall COMET score for subset: 0.8629
